In [1]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Modèle chargé avec succès !")


c:\Users\KOALA\Documents\Enterprise_AI_Copilot\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
c:\Users\KOALA\Documents\Enterprise_AI_Copilot\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KOALA\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might requir

Modèle chargé avec succès !


In [2]:

phrase = "Je souhaite prendre des vacances."

embedding = model.encode(phrase)

print("Dimensions :", embedding.shape)
print("Premières valeurs :", embedding[:10])


Dimensions : (384,)
Premières valeurs : [ 0.36278942  0.04566692  0.12620378  0.0603011  -0.01202425  0.09257635
  0.05475852 -0.08581418 -0.22409832  0.17491569]


Prochaine étape — Similarité cosinus

In [3]:

from sentence_transformers import util

phrase1 = "Je souhaite prendre des vacances."
phrase2 = "Je veux demander un congé annuel."
phrase3 = "Je souhaite consulter une facture."

vecteur1 = model.encode(phrase1)
vecteur2 = model.encode(phrase2)
vecteur3 = model.encode(phrase3)

similarite_12 = util.cos_sim(vecteur1, vecteur2)
similarite_13 = util.cos_sim(vecteur1, vecteur3)

print("Vacances / Congé :", similarite_12.item())
print("Vacances / Facture :", similarite_13.item())


Vacances / Congé : 0.5392909049987793
Vacances / Facture : 0.21216878294944763


In [4]:

from sentence_transformers import util

documents = [
    "Les employés ont droit à 22 jours de congé annuel.",
    "Les factures doivent être payées sous 30 jours.",
    "Le télétravail est autorisé deux jours par semaine.",
    "Les salariés bénéficient d'une assurance maladie.",
    "Les demandes de congé doivent être validées par le responsable."
]

question = "Combien de jours de vacances ont les employés ?"

vecteur_question = model.encode(question)
vecteurs_documents = model.encode(documents)

scores = util.cos_sim(
    vecteur_question,
    vecteurs_documents
)[0]

meilleur_index = scores.argmax().item()

print("Question :", question)
print("Document trouvé :", documents[meilleur_index])
print("Score :", round(scores[meilleur_index].item(), 3))


Question : Combien de jours de vacances ont les employés ?
Document trouvé : Les employés ont droit à 22 jours de congé annuel.
Score : 0.569


Mais dans une vraie entreprise, nous pourrions avoir des milliers de documents.

Nous allons donc utiliser ChromaDB, une base de données vectorielle.

In [5]:

import chromadb

print("ChromaDB installé avec succès !")
print("Version :", chromadb.__version__)


ChromaDB installé avec succès !
Version : 1.5.9


In [6]:

import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_or_create_collection(
    name="documents_rh"
)

print("Collection créée avec succès !")
print("Nom :", collection.name)


Collection créée avec succès !
Nom : documents_rh


In [7]:

documents = [
    "Les employés ont droit à 22 jours de congé annuel.",
    "Les factures doivent être payées sous 30 jours.",
    "Le télétravail est autorisé deux jours par semaine.",
    "Les salariés bénéficient d'une assurance maladie.",
    "Les demandes de congé doivent être validées par le responsable."
]

ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

embeddings = model.encode(documents).tolist()

collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)

print("Documents enregistrés avec succès !")
print("Nombre de documents :", collection.count())


Documents enregistrés avec succès !
Nombre de documents : 5


In [8]:

question = "Combien de jours de vacances ont les employés ?"

vecteur_question = model.encode(question).tolist()

resultats = collection.query(
    query_embeddings=[vecteur_question],
    n_results=2
)

print("Question :", question)

print("\nDocuments retrouvés :")

for i, document in enumerate(resultats["documents"][0]):
    distance = resultats["distances"][0][i]

    print(f"\nDocument {i+1} : {document}")
    print(f"Distance : {distance:.3f}")


Question : Combien de jours de vacances ont les employés ?

Documents retrouvés :

Document 1 : Les employés ont droit à 22 jours de congé annuel.
Distance : 22.563

Document 2 : Le télétravail est autorisé deux jours par semaine.
Distance : 25.570


## Module 3 — Construire notre premier RAG avec Qwen et ChromaDB